# Load Pipeline

In [10]:
import os
import sys

project_root = os.path.abspath("..") 
if project_root not in sys.path:
    sys.path.append(project_root)

# Create preprocessing pipeline
from src.pipeline1.StructuredData import *

# Define column lists
numeric_cols = [
    'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 
    'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 
    'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 
    'Credit_Utilization_Ratio', 'Total_EMI_per_month', 
    'Amount_invested_monthly', 'Monthly_Balance'
]

categorical_cols = ['Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

scaling_cols = ['Credit_History_Age', 'Annual_Income', 'Outstanding_Debt', 
               'Monthly_Inhand_Salary', 'Credit_Utilization_Ratio', 'Total_EMI_per_month', 
               'Amount_invested_monthly', 'Monthly_Balance']

# Create preprocessing pipeline
preprocessing_pipeline = Pipeline([
    ('data_cleaner', DataCleaner()),
    ('numeric_cleaner', NumericCleaner(numeric_cols)),
    ('credit_age_parser', CreditAgeParser()),
    ('loan_encoder', LoanTypeEncoder()),
    ('categorical_encoder', CategoricalEncoder(categorical_cols)),
    ('target_encoder', TargetEncoder()),
])

# Create scaling pipeline (separate because we need to apply after missing value handling)
scaling_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

print("Pipeline created successfully!")

Pipeline created successfully!


# Apply Pipeline

In [ ]:
# Load and preprocess data
data_path = os.path.join("..", "data", "raw", "structured", "kaggle_credit_score_classification", "train.csv")
data = pd.read_csv(data_path)

# Drop personal information
data = data.drop(columns=['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation'])

# Apply preprocessing pipeline
data_preprocessed = preprocessing_pipeline.fit_transform(data)

print(f"Original shape: {data.shape}")
print(f"Preprocessed shape: {data_preprocessed.shape}")

# Handle negative values
invalid_cols = [
    "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts",
    "Num_Credit_Card", "Interest_Rate", "Num_of_Loan", "Delay_from_due_date",
    "Num_of_Delayed_Payment", "Num_Credit_Inquiries", "Outstanding_Debt",
    "Credit_Utilization_Ratio", "Credit_History_Age", "Total_EMI_per_month"
]

for col in invalid_cols:
    if col in data_preprocessed.columns:
        negative_count = (data_preprocessed[col] < 0).sum()
        if negative_count > 0:
            data_preprocessed.loc[data_preprocessed[col] < 0, col] = np.nan # type: ignore

# Handle missing values
for col in data_preprocessed.columns:
    if data_preprocessed[col].dtype == 'object':
        data_preprocessed[col] = data_preprocessed[col].fillna('unknown')
    else:
        data_preprocessed[col] = data_preprocessed[col].fillna(data_preprocessed[col].mean())

# Apply scaling to numeric columns
numeric_features = [col for col in scaling_cols if col in data_preprocessed.columns]
if numeric_features:
    data_preprocessed[numeric_features] = scaling_pipeline.fit_transform(data_preprocessed[numeric_features])

# Handle outliers
for col in data_preprocessed.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'Credit_Score':  # Don't clip target variable
        q1 = data_preprocessed[col].quantile(0.25)
        q3 = data_preprocessed[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        data_preprocessed[col] = data_preprocessed[col].clip(lower=lower_bound, upper=upper_bound)

print("Preprocessing completed!")

# Save final preprocessed training data
data_preprocessed.to_csv(os.path.join("..", "data", "processed", "train_preprocessed.csv"), index=False)
print("Training data saved!")

C:\Users\PC\AppData\Local\Temp\ipykernel_20684\3590945843.py:3: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path)


# Save Pipeline

In [10]:
import joblib

# Save the fitted pipeline
joblib.dump(preprocessing_pipeline, os.path.join("..", "models", "preprocessing_pipeline.pkl"))
joblib.dump(scaling_pipeline, os.path.join("..", "models", "scaling_pipeline.pkl"))

# Save column information
pipeline_info = {
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'scaling_cols': scaling_cols,
    'feature_names': list(data_preprocessed.columns)
}
joblib.dump(pipeline_info, os.path.join("..", "models", "pipeline_info.pkl"))

print("Pipeline saved successfully!")

# Function to apply to test/validation data
def preprocess_new_data(new_data_path, pipeline_path="../models/preprocessing_pipeline.pkl"):
    # Load saved pipeline
    pipeline = joblib.load(pipeline_path)
    scaling_pipe = joblib.load("../models/scaling_pipeline.pkl")
    info = joblib.load("../models/pipeline_info.pkl")
    
    # Load new data
    new_data = pd.read_csv(new_data_path)
    
    # Drop same columns as training
    new_data = new_data.drop(columns=['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation'])
    
    # Apply preprocessing
    processed_data = pipeline.transform(new_data)
    
    # Handle missing values (same strategy as training)
    for col in processed_data.columns:
        if processed_data[col].dtype == 'object':
            processed_data[col] = processed_data[col].fillna('unknown')
        else:
            processed_data[col] = processed_data[col].fillna(processed_data[col].mean())
    
    # Apply scaling
    numeric_features = [col for col in info['scaling_cols'] if col in processed_data.columns]
    if numeric_features:
        processed_data[numeric_features] = scaling_pipe.transform(processed_data[numeric_features])
    
    return processed_data



Pipeline saved successfully!


# Example

In [4]:
# Example: Apply to test data
# test_data = preprocess_new_data("../data/raw/structured/kaggle_credit_score_classification/test.csv")
# print(f"Test data shape: {test_data.shape}")